# Feature Scale and the Geometry of Optimization

This short notebook shows why feature scaling matters in machine learning.

We will study two quadratic functions:

$$
f_{\text{balanced}}(x_1,x_2)=x_1^2+x_2^2
$$

and

$$
f_{\text{stretched}}(x_1,x_2)=100x_1^2+x_2^2.
$$

The second function changes much faster along $x_1$ than along $x_2$.

## Main idea

When one variable or feature operates on a much larger scale, changes along that direction can dominate the output and distort the geometry of the optimization problem.

In [ ]:
import matplotlib.pyplot as plt
import torch

torch.set_default_dtype(torch.float64)

## 1. Define the functions and gradients

For

$$
f(x_1,x_2)=a x_1^2+b x_2^2,
$$

the gradient is

$$
\nabla f(x_1,x_2)=
\begin{bmatrix}
2ax_1 \\
2bx_2
\end{bmatrix}.
$$

In [ ]:
def quadratic(x1, x2, a=1.0, b=1.0):
    return a * x1**2 + b * x2**2


def quadratic_gradient(point, a=1.0, b=1.0):
    x1, x2 = point
    return torch.tensor([2 * a * x1, 2 * b * x2])

## 2. Level curves

Level curves contain all points with the same function value:

$$
f(x_1,x_2)=c.
$$

For the balanced function, the contours are circles.  
For the stretched function, they become narrow ellipses.

In [ ]:
x1 = torch.linspace(-2.0, 2.0, 300)
x2 = torch.linspace(-2.0, 2.0, 300)
X1, X2 = torch.meshgrid(x1, x2, indexing="xy")

Z_balanced = quadratic(X1, X2, a=1.0, b=1.0)
Z_stretched = quadratic(X1, X2, a=100.0, b=1.0)

levels_balanced = [0.25, 0.5, 1, 2, 3, 4]
levels_stretched = [0.25, 0.5, 1, 2, 3, 4]

plt.figure(figsize=(7, 6))
plt.contour(X1, X2, Z_balanced, levels=levels_balanced)
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.title(r"Balanced geometry: $f(x_1,x_2)=x_1^2+x_2^2$")
plt.axis("equal")
plt.show()

plt.figure(figsize=(7, 6))
plt.contour(X1, X2, Z_stretched, levels=levels_stretched)
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.title(r"Stretched geometry: $f(x_1,x_2)=100x_1^2+x_2^2$")
plt.axis("equal")
plt.show()

## 3. The same step does not have the same effect

Start at

$$
(x_1,x_2)=(0,0).
$$

Now compare two changes of equal size:

$$
\Delta x_1=0.5
$$

and

$$
\Delta x_2=0.5.
$$

For the stretched function:

$$
f(0.5,0)=100(0.5)^2=25,
$$

while

$$
f(0,0.5)=(0.5)^2=0.25.
$$

The same numerical change causes a much larger output change along $x_1$.

In [ ]:
step = 0.5

change_x1 = quadratic(
    torch.tensor(step),
    torch.tensor(0.0),
    a=100.0,
    b=1.0,
)

change_x2 = quadratic(
    torch.tensor(0.0),
    torch.tensor(step),
    a=100.0,
    b=1.0,
)

print(f"Output after changing x1 by {step}: {change_x1.item():.2f}")
print(f"Output after changing x2 by {step}: {change_x2.item():.2f}")
print(
    "Ratio:",
    f"{(change_x1 / change_x2).item():.0f} times larger",
)

## 4. Gradient descent paths

Gradient descent updates the parameters using

$$
x^{(t+1)}
=
x^{(t)}
-
\eta\nabla f(x^{(t)}).
$$

We use the same starting point and learning rate for both functions.

In [ ]:
def gradient_descent_path(
    start,
    learning_rate,
    steps,
    a=1.0,
    b=1.0,
):
    point = torch.tensor(start)
    path = [point.clone()]

    for _ in range(steps):
        gradient = quadratic_gradient(point, a=a, b=b)
        point = point - learning_rate * gradient
        path.append(point.clone())

    return torch.stack(path)


start = [1.5, 1.5]
steps = 30

path_balanced = gradient_descent_path(
    start=start,
    learning_rate=0.10,
    steps=steps,
    a=1.0,
    b=1.0,
)

path_stretched = gradient_descent_path(
    start=start,
    learning_rate=0.009,
    steps=steps,
    a=100.0,
    b=1.0,
)

In [ ]:
plt.figure(figsize=(7, 6))
plt.contour(X1, X2, Z_balanced, levels=levels_balanced)
plt.plot(
    path_balanced[:, 0],
    path_balanced[:, 1],
    marker="o",
    markersize=3,
)
plt.scatter(0, 0, marker="*", s=150)
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.title("Gradient descent on balanced contours")
plt.axis("equal")
plt.show()

plt.figure(figsize=(7, 6))
plt.contour(X1, X2, Z_stretched, levels=levels_stretched)
plt.plot(
    path_stretched[:, 0],
    path_stretched[:, 1],
    marker="o",
    markersize=3,
)
plt.scatter(0, 0, marker="*", s=150)
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.title("Gradient descent on stretched contours")
plt.axis("equal")
plt.show()

### Interpretation

On the balanced surface, gradient descent moves smoothly toward the minimum.

On the stretched surface:

- the gradient along $x_1$ is much larger;
- the learning rate must be reduced to remain stable;
- progress along $x_2$ becomes slow;
- the path may zig-zag across the narrow valley.

This is a conditioning problem.

## 5. Rescaling changes the geometry

Define a new coordinate:

$$
z_1=10x_1,
\qquad
z_2=x_2.
$$

Then

$$
100x_1^2+x_2^2
=
z_1^2+z_2^2.
$$

In the scaled coordinates, the stretched ellipses become circles.

In [ ]:
z1 = torch.linspace(-2.0, 2.0, 300)
z2 = torch.linspace(-2.0, 2.0, 300)
Z1, Z2 = torch.meshgrid(z1, z2, indexing="xy")

Z_scaled = Z1**2 + Z2**2

path_scaled = gradient_descent_path(
    start=[15.0, 1.5],
    learning_rate=0.10,
    steps=30,
    a=1.0,
    b=1.0,
)

plt.figure(figsize=(7, 6))
plt.contour(Z1, Z2, Z_scaled, levels=levels_balanced)
plt.plot(
    path_scaled[:, 0],
    path_scaled[:, 1],
    marker="o",
    markersize=3,
)
plt.scatter(0, 0, marker="*", s=150)
plt.xlabel("$z_1=10x_1$")
plt.ylabel("$z_2=x_2$")
plt.title("After rescaling: balanced geometry again")
plt.axis("equal")
plt.show()

## 6. Connection to linear regression

Consider

$$
\hat{y}=w_1x_1+w_2x_2+b.
$$

A small change in a weight changes the prediction by

$$
\Delta \hat{y}=x_j\Delta w_j.
$$

Therefore, if $x_1$ typically has values around $1000$ and $x_2$ has values around $1$, the same parameter update produces:

$$
\Delta \hat{y}_1\approx1000\Delta w_1,
$$

but

$$
\Delta \hat{y}_2\approx\Delta w_2.
$$

The large-scale feature has a stronger numerical effect on both predictions and gradients.

Standardization,

$$
z_j=\frac{x_j-\mu_j}{\sigma_j},
$$

places features on comparable scales and usually makes the loss contours less elongated.

# Short activity

Change the coefficient `100` in

```python
f(x1, x2) = 100 * x1**2 + x2**2
```

to:

- `4`
- `25`
- `400`

For each value:

1. Plot the contours.
2. Run gradient descent.
3. Find the largest learning rate that remains stable.
4. Describe how the path changes.

## Final question

Why can standardization allow gradient descent to use a larger learning rate and converge in fewer steps?